# Real Number Generator Using DCGAN (Deep Convolutional Generative Adversarial Network)

In a GAN, two models "fight" each other:

- The Generator: Tries to create fake digits from random noise.

- The Discriminator: Tries to tell if a digit is real (from your SVHN dataset) or fake (from the Generator).

# Data Analysis And Setup

Since SVHN images are $32 \times 32$ pixels and in color (RGB), the generator must produce three channels. We normalize pixel values to $[-1, 1]$ to match the Tanh activation of the generator.

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import numpy as np

# MNIST is grayscale (1 channel), so we use a single mean/std value
transform = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(root='./data', train=True, download=True, transform=transform),
    batch_size=128, shuffle=True
)

100%|██████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:08<00:00, 1.23MB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 70.5kB/s]
100%|███████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:04<00:00, 387kB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 15.0MB/s]


# The Generator

The Generator takes a "latent vector" (random noise) and uses Transposed Convolutions (upsampling) to grow it into a $32 \times 32$ image.

In [3]:
class Generator(nn.Module):
    def __init__(self, nz=100, ngf=64):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            # Input is nz (100-dim noise)
            nn.ConvTranspose2d(nz, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size: (ngf*4) x 4 x 4
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size: (ngf*2) x 8 x 8
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # FINAL LAYER CHANGE: Output 1 channel for MNIST
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False),
            nn.Tanh()
            # final size: 1 x 32 x 32
        )

    def forward(self, input):
        return self.main(input)

# The Discriminator

The Discriminator is essentially a binary classifier (Real vs. Fake).

In [4]:
class Discriminator(nn.Module):
    def __init__(self, ndf=64):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            # INPUT CHANGE: 1 channel input instead of 3
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state size: (ndf) x 16 x 16
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            # state size: (ndf*2) x 8 x 8
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            # state size: (ndf*4) x 4 x 4
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),
            # Remember to remove Sigmoid if using BCEWithLogitsLoss!
        )

    def forward(self, input):
        return self.main(input).view(-1, 1).squeeze(1)

# Training and Evaluation

In [5]:
# --- Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nz = 100  # Size of latent vector
epochs = 25
lr = 0.0002
beta1 = 0.5
scaler = torch.cuda.amp.GradScaler() # For Mixed Precision on your 3050

# Fixed noise to track the evolution of specific digits
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# Initialize Models (Assuming Generator/Discriminator classes are defined above)
netG = Generator().to(device)
netD = Discriminator().to(device)
criterion = nn.BCEWithLogitsLoss()

optimizerD = torch.optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = torch.optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

print("Starting Training Loop...")

for epoch in range(epochs):
    for i, (data, _) in enumerate(train_loader):
        
        ############################
        # (1) Update Discriminator: maximize log(D(x)) + log(1 - D(G(z)))
        ############################
        netD.zero_grad()
        real_cpu = data.to(device)
        batch_size = real_cpu.size(0)
        label = torch.full((batch_size,), 1.0, dtype=torch.float, device=device)

        # Forward pass real batch through D
        with torch.cuda.amp.autocast(): # Mixed precision context
            output = netD(real_cpu).view(-1)
            errD_real = criterion(output, label)
        
        scaler.scale(errD_real).backward()

        # Generate fake image batch
        noise = torch.randn(batch_size, nz, 1, 1, device=device)
        with torch.cuda.amp.autocast():
            fake = netG(noise)
            label.fill_(0.0) # Label as fake
            output = netD(fake.detach()).view(-1)
            errD_fake = criterion(output, label)
        
        scaler.scale(errD_fake).backward()
        scaler.step(optimizerD)
        
        ############################
        # (2) Update Generator: maximize log(D(G(z)))
        ############################
        netG.zero_grad()
        label.fill_(1.0) # Fake labels are real for generator cost
        
        with torch.cuda.amp.autocast():
            output = netD(fake).view(-1)
            errG = criterion(output, label)
        
        scaler.scale(errG).backward()
        scaler.step(optimizerG)
        
        # Update scaler for next iteration
        scaler.update()

        # Progress Monitoring
        if i % 100 == 0:
            print(f'[{epoch}/{epochs}][{i}/{len(train_loader)}] '
                  f'Loss_D: {errD_real.item()+errD_fake.item():.4f} '
                  f'Loss_G: {errG.item():.4f}')

    # --- SAVE PROGRESS SNAPSHOT ---
    with torch.no_grad():
        fake_display = netG(fixed_noise).detach().cpu()
    
    plt.figure(figsize=(8,8))
    plt.axis("off")
    plt.title(f"Generated Digits at Epoch {epoch}")
    grid = vutils.make_grid(fake_display, padding=2, normalize=True)
    plt.imshow(np.transpose(grid,(1,2,0)))
    plt.savefig(f"svhn_gan_epoch_{epoch}.png")
    plt.close()

# Save final model weights
torch.save(netG.state_dict(), "final_generator.pth")
print("Training Complete. Snapshots saved to directory.")

/tmp/ipykernel_43989/3523547767.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # For Mixed Precision on your 3050
/tmp/ipykernel_43989/3523547767.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): # Mixed precision context


Starting Training Loop...


/tmp/ipykernel_43989/3523547767.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_43989/3523547767.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[0/25][0/469] Loss_D: 1.5182 Loss_G: 1.0160
[0/25][100/469] Loss_D: 0.0037 Loss_G: 6.4295
[0/25][200/469] Loss_D: 0.0422 Loss_G: 5.6114
[0/25][300/469] Loss_D: 0.7323 Loss_G: 2.4858
[0/25][400/469] Loss_D: 0.4310 Loss_G: 2.5693
[1/25][0/469] Loss_D: 0.5479 Loss_G: 2.1832
[1/25][100/469] Loss_D: 0.4554 Loss_G: 1.1607
[1/25][200/469] Loss_D: 0.6726 Loss_G: 1.4236
[1/25][300/469] Loss_D: 0.4274 Loss_G: 2.7577
[1/25][400/469] Loss_D: 0.1957 Loss_G: 2.6660
[2/25][0/469] Loss_D: 0.3088 Loss_G: 3.0558
[2/25][100/469] Loss_D: 1.6643 Loss_G: 4.4192
[2/25][200/469] Loss_D: 0.8704 Loss_G: 2.4005
[2/25][300/469] Loss_D: 0.6033 Loss_G: 3.0589
[2/25][400/469] Loss_D: 0.2933 Loss_G: 1.7348
[3/25][0/469] Loss_D: 0.3019 Loss_G: 2.9811
[3/25][100/469] Loss_D: 0.3026 Loss_G: 2.4786
[3/25][200/469] Loss_D: 0.3486 Loss_G: 2.4809
[3/25][300/469] Loss_D: 0.2264 Loss_G: 2.4174
[3/25][400/469] Loss_D: 0.6380 Loss_G: 4.4427
[4/25][0/469] Loss_D: 0.1789 Loss_G: 2.7681
[4/25][100/469] Loss_D: 0.1496 Loss_G: 3.245